In [ ]:
# imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pmdarima import auto_arima

In [ ]:
# 1 sliding windows

def make_sliding_windows(y, history=28, horizon=14, step=1):
    y = np.asarray(y)
    n = len(y)

    X, Y, origins = [], [], []

    for t in range(history, n - horizon + 1, step):
        X.append(y[t-history:t])
        Y.append(y[t:t+horizon])
        origins.append(t)

    return np.array(X), np.array(Y), np.array(origins)

In [ ]:
# 2 time-ordered Train/Test Split

def split_train_test_windows(X, Y, origins, test_ratio=0.2):
    n = len(X)
    split = int(n * (1 - test_ratio))

    return (
        X[:split], Y[:split],
        X[split:], Y[split:],
        origins[:split], origins[split:]
    )

# 3 AutoARIMA ROlling Forecast
def run_autoarima_on_test_windows(
    X_test,
    Y_test,
    seasonal=False,
    m=1,
    alpha=0.1,
):
    results = []

    for i in range(len(X_test)):
        hist = X_test[i]
        y_true = Y_test[i]

        
        
        model = auto_arima(
            hist,
            seasonal=seasonal,
            m=m,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore",
            trace=False,
            n_fits=30,

            start_q=0,
            max_q=0,        # 🔴 THIS is the key
        )

        y_pred, conf = model.predict(
            n_periods=len(y_true),
            return_conf_int=True,
            alpha=alpha,
        )

        results.append({
            "sample_id": i,
            "history": hist,
            "y_true": y_true,
            "y_pred": y_pred,
            "lower": conf[:, 0],
            "upper": conf[:, 1],
            "order": model.order,
            "seasonal_order": model.seasonal_order,
        })

    return results

In [ ]:
# 4 Metrics

# RMSE + Coverage
# Coverage measures how often the true value falls inside your predicted interval
# Coverage = percentage of times your prediction interval captures the true value
def evaluate_forecasts(results):
    y_true, y_pred, lower, upper = [], [], [], []

    for r in results:
        y_true.extend(r["y_true"])
        y_pred.extend(r["y_pred"])
        lower.extend(r["lower"])
        upper.extend(r["upper"])

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    lower = np.array(lower)
    upper = np.array(upper)

    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    coverage = np.mean((y_true >= lower) & (y_true <= upper))

    return rmse, coverage


In [ ]:
# Winkler Score
def compute_winkler(results, alpha=0.1):
    scores = []

    for r in results:
        for y, l, u in zip(r["y_true"], r["lower"], r["upper"]):
            width = u - l
            if y < l:
                score = width + 2/alpha * (l - y)
            elif y > u:
                score = width + 2/alpha * (y - u)
            else:
                score = width
            scores.append(score)

    return np.mean(scores)

In [ ]:
# Horizon-wise Evaluation
def evaluate_by_horizon(results):
    H = len(results[0]["y_true"])

    rmse_h, coverage_h = [], []

    for h in range(H):
        yt, yp, l, u = [], [], [], []

        for r in results:
            yt.append(r["y_true"][h])
            yp.append(r["y_pred"][h])
            l.append(r["lower"][h])
            u.append(r["upper"][h])

        yt, yp, l, u = map(np.array, [yt, yp, l, u])

        rmse_h.append(np.sqrt(np.mean((yt - yp)**2)))
        coverage_h.append(np.mean((yt >= l) & (yt <= u)))

    return rmse_h, coverage_h

In [ ]:
# Coverage vs Forecast Horizon (Train vs Test)
def plot_coverage_train_vs_test(
    coverage_h_train,
    coverage_h_test,
    dataset_name,
    target=0.9
):
    h = range(1, len(coverage_h_test) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(h, coverage_h_train, marker="o", label="Train Coverage")
    plt.plot(h, coverage_h_test, marker="o", label="Test Coverage")

    plt.axhline(target, linestyle="--", color="red",
                label=f"Target {int(target*100)}%")

    plt.xlabel("Forecast Horizon")
    plt.ylabel("Coverage")
    plt.title(f"Coverage vs Forecast Horizon ({dataset_name})")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
def compute_sample_rmse(results):
    """
    Compute RMSE for each test window.
    Returns a sorted list of (sample_id, rmse).
    """
    errors = []
    for i, r in enumerate(results):
        y_true = np.array(r["y_true"])
        y_pred = np.array(r["y_pred"])
        rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
        errors.append((i, rmse))
    return sorted(errors, key=lambda x: x[1])

In [ ]:
# 5 Forecast Plot(Single Window)
def plot_single_forecast(results, sample_id=0, model_name="AutoARIMA"):
    r = results[sample_id]

    hist = r["history"]
    y_true = r["y_true"]
    y_pred = r["y_pred"]

    full = np.concatenate([hist, y_true])
    t_full = np.arange(len(full))
    t_pred = np.arange(len(hist), len(hist) + len(y_true))

    plt.figure(figsize=(10, 4))
    plt.plot(t_full, full, label="True", color="black")
    plt.plot(t_pred, y_pred, label="Forecast", color="red")
    plt.fill_between(t_pred, r["lower"], r["upper"], alpha=0.3)

    plt.axvline(len(hist) - 1, linestyle="--", color="gray")
    plt.title(
        f"{model_name} ARIMA{r['order']} | Horizon={len(y_true)}"
    )
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# plot the best/worst forecasts plots
def plot_best_worst_forecasts(results, dataset_name, n_show=3):
    """
    Plot the best and worst AutoARIMA forecasts for one dataset.
    """
    errors = compute_sample_rmse(results)

    best = errors[:n_show]
    worst = errors[-n_show:]

    print(f"\n📈 Best {n_show} forecasts for {dataset_name}")
    for sid, err in best:
        print(f"Sample {sid}, RMSE={err:.3f}")
        plot_single_forecast(results, sid, model_name="AutoARIMA (Best)")

    print(f"\n📉 Worst {n_show} forecasts for {dataset_name}")
    for sid, err in worst:
        print(f"Sample {sid}, RMSE={err:.3f}")
        plot_single_forecast(results, sid, model_name="AutoARIMA (Worst)")

In [ ]:
def run_autoarima_pipeline(
    y,
    test_ratio=0.2,
    seasonal=False,
    m=1,
    alpha=0.1,
):
    # -----------------------------
    # Scale-aware history & horizon
    # -----------------------------
    history = min(100, len(y) // 3)
    horizon = min(28, len(y) // 6)

    # -----------------------------
    # SAFETY CHECK (THIS IS THE KEY)
    # -----------------------------
    if len(y) < history + horizon + 1:
        return None
    
    # --------------------------------------------------
    # 1. Create sliding windows
    # --------------------------------------------------
    X, Y, origins = make_sliding_windows(y, history, horizon)

    # --------------------------------------------------
    # 2. Train / Test split (time-ordered)
    # --------------------------------------------------
    X_train, Y_train, X_test, Y_test, _, _ = split_train_test_windows(
        X, Y, origins, test_ratio
    )

    # --------------------------------------------------
    # 3. TEST: Run AutoARIMA on TEST windows
    # --------------------------------------------------
    results_test = run_autoarima_on_test_windows(
        X_test,
        Y_test,
        seasonal=seasonal,
        m=m,
        alpha=alpha,
    )

    # Global test metrics
    rmse, coverage = evaluate_forecasts(results_test)
    winkler = compute_winkler(results_test, alpha)

    # Horizon-wise test metrics
    rmse_h_test, coverage_h_test = evaluate_by_horizon(results_test)

    # --------------------------------------------------
    # 4. TRAIN: Run AutoARIMA on TRAIN windows (diagnostics)
    # --------------------------------------------------
    results_train = run_autoarima_on_test_windows(
        X_train,
        Y_train,
        seasonal=seasonal,
        m=m,
        alpha=alpha,
    )

    # Horizon-wise train coverage (for comparison)
    _, coverage_h_train = evaluate_by_horizon(results_train)

    # --------------------------------------------------
    # 5. Return everything needed for plots & tables
    # --------------------------------------------------
    return {
        "results_test": results_test,
        "results_train": results_train,
        "rmse": rmse,
        "coverage": coverage,
        "winkler": winkler,
        "rmse_h_test": rmse_h_test,
        "coverage_h_test": coverage_h_test,
        "coverage_h_train": coverage_h_train,
    }

    

In [ ]:
def parse_key(key):
    """
    Parse dataset key into metadata.
    Example key:
    'sawtooth_n1000_seed1_obs_fast'
    """
    parts = key.split("_")

    meta = {
        "generator": parts[0],
        "obs_scale": "latent",
        "sample_size": None,
    }

    for p in parts:
        if p.startswith("n") and p[1:].isdigit():
            meta["sample_size"] = int(p[1:])
        if p.startswith("obs"):
            meta["obs_scale"] = p

    return meta

In [ ]:
# --------------------------------------------------
# 7. Run on ALL Synthetic Datasets
# --------------------------------------------------
data = np.load("/synthetic_time_series.npz")
summary = []

for key in data.files:
    print(f"Running AutoARIMA on {key}")

    y = data[key]

    # Run pipeline
    out = run_autoarima_pipeline(y)

    # Skip datasets that are too short
    if out is None:
        print(f"Skipping {key} (too short for history+horizon)")
        continue

    # Parse metadata from key
    meta = parse_key(key)

    # ---- Best / Worst forecast plots ----
    plot_best_worst_forecasts(
        out["results_test"],
        dataset_name=key
    )

    # ---- Coverage Train vs Test ----
    plot_coverage_train_vs_test(
        out["coverage_h_train"],
        out["coverage_h_test"],
        dataset_name=key
    )

    # ---- Collect summary metrics ----
    summary.append({
        "dataset": key,
        "generator": meta["generator"],
        "obs_scale": meta["obs_scale"],
        "sample_size": meta["sample_size"],
        "RMSE": out["rmse"],
        "Coverage": out["coverage"],
        "Winkler": out["winkler"],
    })

# --------------------------------------------------
# Summary table
# --------------------------------------------------
df = pd.DataFrame(summary)
print(df)


In [ ]:
# Summary Table
df = pd.DataFrame(summary)
print(df)
